In [28]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import sqlite3
import os

BASE     = '/content/drive/MyDrive/ecommerce-etl-pipeline'
DB_PATH  = f'{BASE}/data/sql/ecommerce.db'
SQL_PATH = f'{BASE}/data/sql'

conn = sqlite3.connect(DB_PATH)

def run_query(query, title=""):
    result = pd.read_sql(query, conn)
    if title:
        print(f"\n{'='*40}")
        print(f" {title}")
        print(f"{'='*40}")
    print(result.to_string(index=False))
    return result

print("Connected!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Connected!


In [29]:
conn.execute("DROP VIEW IF EXISTS vw_monthly_revenue;")

conn.execute("""
CREATE VIEW vw_monthly_revenue AS
SELECT
    order_year,
    order_month,
    order_month_year,
    COUNT(order_id)                             AS total_orders,
    ROUND(SUM(total_revenue), 2)                AS monthly_revenue,
    ROUND(AVG(total_revenue), 2)                AS avg_order_value,
    ROUND(SUM(SUM(total_revenue))
          OVER (ORDER BY order_month_year), 2)  AS running_total
FROM master_orders
GROUP BY order_year, order_month, order_month_year
ORDER BY order_month_year;
""")

conn.commit()
print("View created!")

View created!


In [30]:
kpi_monthly = run_query(
    "SELECT * FROM vw_monthly_revenue;",
    "KPI 1 — Monthly Revenue Trend"
)

kpi_monthly.to_csv(f'{SQL_PATH}/kpi_monthly_revenue.csv', index=False)
print("Exported!")


 KPI 1 — Monthly Revenue Trend
 order_year  order_month order_month_year  total_orders  monthly_revenue  avg_order_value  running_total
       2016            9          2016-09             1              NaN              NaN            NaN
       2016           10          2016-10           265         46566.71           175.72       46566.71
       2016           12          2016-12             1            19.62            19.62       46586.33
       2017            1          2017-01           750        127545.67           170.06      174132.00
       2017            2          2017-02          1653        271298.65           164.13      445430.65
       2017            3          2017-03          2546        414369.39           162.75      859800.04
       2017            4          2017-04          2303        390952.18           169.76     1250752.22
       2017            5          2017-05          3545        566872.73           159.91     1817624.95
       2017            

In [31]:
conn.execute("DROP VIEW IF EXISTS vw_top_products;")

conn.execute("""
CREATE VIEW vw_top_products AS
SELECT
    product_category_name_english       AS category,
    COUNT(order_id)                     AS total_orders,
    ROUND(SUM(total_revenue), 2)        AS total_revenue,
    ROUND(AVG(total_revenue), 2)        AS avg_order_value,
    RANK() OVER (
        ORDER BY SUM(total_revenue) DESC
    )                                   AS revenue_rank
FROM master_orders
WHERE product_category_name_english != 'Unknown'
GROUP BY product_category_name_english
ORDER BY total_revenue DESC;
""")

conn.commit()
print("View created!")

# Query the view and store in a DataFrame
kpi_products = run_query(
    "SELECT * FROM vw_top_products;",
    "KPI 2 — Top Product Categories"
)

# Export to CSV
kpi_products.to_csv(f'{SQL_PATH}/kpi_top_products.csv', index=False)
print("Exported!")

View created!

 KPI 2 — Top Product Categories
                               category  total_orders  total_revenue  avg_order_value  revenue_rank
                          health_beauty          8608     1410846.79           163.92             1
                          watches_gifts          5470     1261634.89           230.65             2
                         bed_bath_table          9167     1224487.19           133.58             3
                         sports_leisure          7490     1119160.77           149.42             4
                  computers_accessories          6500     1030732.32           158.57             5
                        furniture_decor          6213      884202.65           142.31             6
                             housewares          5688      760768.67           133.75             7
                             cool_stuff          3530      693002.80           196.32             8
                                   auto          3792

In [32]:
# Drop view if it exists
conn.execute("DROP VIEW IF EXISTS vw_regional_performance;")

# Create the view for regional performance
conn.execute("""
CREATE VIEW vw_regional_performance AS
SELECT
    customer_state,
    COUNT(order_id)                                     AS total_orders,
    ROUND(SUM(total_revenue), 2)                        AS total_revenue,
    ROUND(AVG(total_revenue), 2)                        AS avg_order_value,
    ROUND(AVG(delivery_days), 2)                        AS avg_delivery_days,
    ROUND(SUM(total_revenue) * 100.0 /
          SUM(SUM(total_revenue)) OVER (), 2)           AS revenue_pct
FROM master_orders
GROUP BY customer_state
ORDER BY total_revenue DESC;
""")

conn.commit()
print("View created!")

# Query the view and store in a DataFrame
kpi_regional = run_query(
    "SELECT * FROM vw_regional_performance;",
    "KPI 3 — Regional Performance"
)

# Export to CSV
kpi_regional.to_csv(f'{SQL_PATH}/kpi_regional_performance.csv', index=False)
print("Exported!")

View created!

 KPI 3 — Regional Performance
customer_state  total_orders  total_revenue  avg_order_value  avg_delivery_days  revenue_pct
            SP         40494     5769081.27           142.47               8.30        37.41
            RJ         12350     2055690.45           166.45              14.85        13.33
            MG         11354     1819277.61           160.23              11.54        11.80
            RS          5344      861608.40           161.23              14.82         5.59
            PR          4923      781919.55           158.83              11.53         5.07
            SC          3546      595208.40           167.85              14.48         3.86
            BA          3256      591270.60           181.59              18.87         3.83
            DF          2080      346146.17           166.42              12.51         2.24
            GO          1957      334294.22           170.82              15.15         2.17
            ES          1

In [33]:
conn.execute("DROP VIEW IF EXISTS vw_customer_retention;")

conn.execute("""
CREATE VIEW vw_customer_retention AS
WITH customer_orders AS (
    SELECT
        c.customer_unique_id,
        COUNT(o.order_id) AS total_orders
    FROM customers c
    JOIN master_orders o
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
)
SELECT
    CASE
        WHEN total_orders = 1 THEN 'One-time buyer'
        WHEN total_orders = 2 THEN 'Returning buyer'
        WHEN total_orders >= 3 THEN 'Loyal buyer'
    END                     AS customer_type,
    COUNT(*)                AS total_customers,
    ROUND(COUNT(*) * 100.0 /
          SUM(COUNT(*)) OVER(), 2) AS pct_of_customers
FROM customer_orders
GROUP BY customer_type
ORDER BY total_customers DESC;
""")

conn.commit()

kpi_retention = run_query(
    "SELECT * FROM vw_customer_retention;",
    "KPI 4 — Customer Retention Fixed"
)

kpi_retention.to_csv(
    f'{SQL_PATH}/kpi_customer_retention.csv', index=False)
print("Exported!")


 KPI 4 — Customer Retention Fixed
  customer_type  total_customers  pct_of_customers
 One-time buyer            90549             97.00
Returning buyer             2573              2.76
    Loyal buyer              228              0.24
Exported!


In [34]:
# Drop view if it exists
conn.execute("DROP VIEW IF EXISTS vw_delivery_performance;")

# Create the view for delivery performance
conn.execute("""
CREATE VIEW vw_delivery_performance AS
SELECT
    order_month_year,
    COUNT(order_id)                                     AS total_orders,
    ROUND(AVG(delivery_days), 2)                        AS avg_delivery_days,
    SUM(CASE WHEN delivered_on_time = 1 THEN 1 ELSE 0 END) AS on_time_deliveries,
    SUM(CASE WHEN delivered_on_time = 0 THEN 1 ELSE 0 END) AS late_deliveries,
    ROUND(
        SUM(CASE WHEN delivered_on_time = 1 THEN 1.0 ELSE 0.0 END) * 100.0 /
        COUNT(order_id), 2
    )                                                   AS on_time_pct
FROM master_orders
GROUP BY order_month_year
ORDER BY order_month_year;
""")

conn.commit()
print("View created!")

# Query the view and store in a DataFrame
kpi_delivery = run_query(
    "SELECT * FROM vw_delivery_performance;",
    "KPI 5 — Delivery Performance"
)

# Export to CSV
kpi_delivery.to_csv(f'{SQL_PATH}/kpi_delivery_performance.csv', index=False)
print("Exported!")

View created!

 KPI 5 — Delivery Performance
order_month_year  total_orders  avg_delivery_days  on_time_deliveries  late_deliveries  on_time_pct
         2016-09             1              54.00                   0                1         0.00
         2016-10           265              19.14                 262                3        98.87
         2016-12             1               4.00                   1                0       100.00
         2017-01           750              12.09                 727               23        96.93
         2017-02          1653              12.61                1600               53        96.79
         2017-03          2546              12.40                2404              142        94.42
         2017-04          2303              14.35                2122              181        92.14
         2017-05          3545              10.76                3417              128        96.39
         2017-06          3135              11.51      

In [35]:
print("=== KPI FILES EXPORTED ===")
for f in os.listdir(SQL_PATH):
    if f.startswith('kpi_'):
        size = os.path.getsize(f'{SQL_PATH}/{f}')
        print(f"{f} — {size:,} bytes")

conn.close()
print("\nAll done!")

=== KPI FILES EXPORTED ===
kpi_monthly_revenue.csv — 1,170 bytes
kpi_top_products.csv — 2,887 bytes
kpi_regional_performance.csv — 1,030 bytes
kpi_customer_retention.csv — 120 bytes
kpi_delivery_performance.csv — 841 bytes

All done!
